## Cell 1: Import all required libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller


## Cell 2: Load your price data

In [2]:
prices = pd.read_csv("data/prices.csv", index_col=0, parse_dates=True)


## Cell 3: Select your two stocks

In [3]:
A = prices["TATASTEEL.NS"]
B = prices["HINDALCO.NS"]


## Cell 4: Run Regression to Find the Hedge Ratio (β) and Intercept (α)

OLS regression of A on B gives the **cointegrating vector**: `A = α + β·B + ε`

Both α and β must be retained to form the stationary spread (OLS residual).
Dropping α shifts the spread mean away from zero and inflates Z-score noise.

In [ ]:
import statsmodels.api as sm

model = sm.OLS(A, sm.add_constant(B)).fit()
beta  = model.params.iloc[1]   # hedge ratio
alpha = model.params.iloc[0]   # intercept

print(f"Hedge Ratio  β = {beta:.6f}")
print(f"Intercept    α = {alpha:.6f}")
print(f"R²           = {model.rsquared:.4f}")

## Cell 5: Create the OLS-Residual Spread

`spread = A − β·B − α`

This is the cointegration residual — the same quantity the Engle–Granger test
checks for stationarity.  Omitting α is a common error that biases the spread
away from zero and produces unreliable Z-scores.

In [ ]:
spread = A - beta * B - alpha      # full OLS residual (mean-zero by construction)

spread.plot(figsize=(12, 5), title="Cointegration Residual Spread: TATASTEEL − β·HINDALCO − α")
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.ylabel("Spread (INR)")
plt.show()

## Cell 6: ADF test for Stationarity 

In [7]:
adf_result = adfuller(spread.dropna())
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])


ADF Statistic: -4.688700016958121
p-value: 8.836360198449679e-05


## Cell 7: Compute Rolling Z-Score (Look-Ahead Free)

A Z-score computed from the **entire sample's** mean and standard deviation contains
look-ahead bias — at any historical date, the normalisation uses future data that would
not yet be observable.

The correct approach is a **rolling window** Z-score:

`z_t = (spread_t − μ_{t-ROLL:t}) / σ_{t-ROLL:t}`

A 252-day (one-year) lookback is standard in equity stat-arb; `min_periods=60` allows
signals to begin after 3 months of data.

In [ ]:
ROLL = 252      # 1-year lookback window

roll_mean = spread.rolling(ROLL, min_periods=60).mean()
roll_std  = spread.rolling(ROLL, min_periods=60).std()
zscore    = (spread - roll_mean) / roll_std

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(spread, color='steelblue', linewidth=0.8)
axes[0].set_title("Cointegration Residual Spread")
axes[0].set_ylabel("Spread (INR)")
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')

axes[1].plot(zscore, color='darkgreen', linewidth=0.8)
axes[1].axhline( 2.0, color='red',   linestyle='--', label='+2σ entry')
axes[1].axhline(-2.0, color='green', linestyle='--', label='−2σ entry')
axes[1].axhline( 0.5, color='orange', linestyle=':', linewidth=0.8, label='±0.5σ exit')
axes[1].axhline(-0.5, color='orange', linestyle=':', linewidth=0.8)
axes[1].set_title(f"Rolling Z-Score (lookback = {ROLL} days)")
axes[1].set_ylabel("Z-Score")
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"Z-score NaN rows (warm-up period): {zscore.isna().sum()}")

## Cell 8: Save Spread & Rolling Z-Score

In [ ]:
spread.to_csv("data/spread_tatasteel_hindalco.csv")
zscore.to_csv("data/zscore_tatasteel_hindalco.csv")
print("Saved spread and rolling Z-score (252-day lookback, look-ahead free).")